# Local LLM Inference with Transformers

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Run modern language models locally on Mac (Intel/Apple Silicon) or PC
* Work with efficient models optimized for CPU/Apple Silicon
* Understand practical limits of local inference
* Master text generation without cloud dependencies
* Implement efficient processing for research tasks

</div>

This notebook shows you how to run open-source LLMs locally on your laptop using the **Transformers** library.

Unlike Ollama, which abstracts everything, Transformers lets you fully control models, tokenizers, quantization formats, and inference pipelines.

## What is Transformers?

Transformers (by Hugging Face) is the most widely used library for loading, running, fine-tuning, and analyzing LLMs.

It provides a unified interface for:
- loading models from Hugging Face
- choosing quantization formats (GGUF, AWQ, GPTQ, etc.)
- running models on CPU, GPU, or Apple Silicon
- integrating with PyTorch or MPS

Key Advantages of Transformers:
- Full Control — you choose model, quantization, device, and pipeline
- Huge Model Ecosystem — all open models on Hugging Face are supported
- Research-Friendly — easily extract logits, embeddings, attention, etc.
- Fine-Tuning Support — LoRA/QLoRA, full-precision training, etc.
- Flexible Backends — run models through PyTorch, MPS, or accelerate

### Best Models for Local Inference with Transformers
*Updated 11/09/2025*

| Model | Size | RAM Needed | Notable Features |
|-------|------|------------|-----------------|
| Qwen3Guard-Gen-0.6B | 0.6B | 2GB | Latest safety-focused model, 119 languages |
| TinyLlama-1.1B | 1.1B | 3GB | Fastest, good for basic tasks |
| Qwen3-4B | 4B | 7GB | Latest Alibaba model, thinking/non-thinking modes |
| Phi-2 | 2.7B | 6GB | Microsoft, excellent reasoning |
| Yi-6B | 6B | 8GB | Strong bilingual, excellent for code |
| Neural-Chat-7B-v3-1 | 7B | 9GB | Optimized for chat, Intel |
| Qwen2.5-7B | 7B | 9GB | Stable Qwen model, strong general performance |
| GPT-OSS-20B | 20B | 16GB | Strong general performance |

All models are open-weights and available on Hugging Face. Models are listed from smallest to largest, with specialized models noted for their strengths. The 0.6B-4B models are particularly suitable for local inference on laptops.

<a id='setup'></a>

# Hardware Detection

Let's check what hardware you have available for local inference.

In [1]:
import platform
import psutil
import torch

def detect_hardware():
    """Detect local hardware capabilities"""
    print("System Information")
    print("=" * 50)
    
    # Operating System
    system = platform.system()
    print(f"OS: {system} {platform.release()}")
    print(f"Python: {platform.python_version()}")
    
    # CPU
    print(f"\nCPU: {platform.processor()}")
    print(f"Cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical")
    
    # Memory
    ram = psutil.virtual_memory().total / (1024**3)
    available_ram = psutil.virtual_memory().available / (1024**3)
    print(f"\nRAM: {ram:.1f} GB total")
    print(f"Available: {available_ram:.1f} GB")
    
    # Check for acceleration
    print("\nAcceleration:")
    
    # Apple Silicon (MPS)
    if torch.backends.mps.is_available():
        print("Apple Silicon detected (MPS acceleration available)")
        device = "mps"
    # NVIDIA GPU
    elif torch.cuda.is_available():
        print(f"NVIDIA GPU detected: {torch.cuda.get_device_name(0)}")
        device = "cuda"
    # CPU only
    else:
        print("ℹUsing CPU (no GPU acceleration detected)")
        device = "cpu"
    
    return device

# Detect hardware
DEVICE = detect_hardware()

System Information
OS: Darwin 24.6.0
Python: 3.10.14

CPU: arm
Cores: 10 physical, 10 logical

RAM: 16.0 GB total
Available: 4.6 GB

Acceleration:
Apple Silicon detected (MPS acceleration available)


<a id='install'></a>

# Installation
Uncomment if needed.

In [ ]:
# Install only essential packages
#!pip install -q torch transformers accelerate
#!pip install -q psutil  # For system monitoring

# Optional: visualization
# !pip install -q matplotlib pandas

# Verify versions
import transformers
import torch
print(f"\nTransformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")


Transformers version: 4.52.4
PyTorch version: 2.7.1


<a id='load'></a>

# Loading Models

Load a model appropriate for your hardware. We'll use standard transformers library - simple and reliable!

For this example, we will use [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).

An “Instruct model” (short for instruction-tuned model) is a large language model that has been fine-tuned to follow human instructions in natural language.

It starts from a base model (which just predicts the next token in raw text), and then goes through an extra supervised fintuning phase (*instruction tuning*) where the model learns to interpret instructions (“summarize”, “explain”, “compare”), and to respond in helpful, complete sentences.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Choose your model (change this based on your RAM)
# With 2.4GB RAM available, we'll use the most efficient option
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  

# Alternative models by RAM requirement:
# 2-3GB RAM:  "Qwen/Qwen3Guard-Gen-0.6B" (0.6B) - Efficient safety classifier / guardrail model
# 3-4GB RAM:  "TinyLlama-1.1B" (1.1B) - Fast, good for basic tasks
# 6-7GB RAM:  "microsoft/phi-2" (2.7B) - Strong reasoning
# 7-8GB RAM:  "Qwen/Qwen3-4B" (4B) - Latest Qwen, thinking mode
# 8-9GB RAM:  "01-ai/Yi-6B" (6B) - Excellent for code
# 9-10GB RAM: "Qwen/Qwen2.5-7B" (7B) - Stable performance

print(f"Loading {MODEL_NAME}...")

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Load model with appropriate settings
    if DEVICE == "mps":  # Apple Silicon
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            trust_remote_code=True
        ).to(DEVICE)
    elif DEVICE == "cuda":  # NVIDIA GPU
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    else:  # CPU
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,  # Full precision for CPU
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )
    
    # Set pad token if needed
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print(f"✅ Model loaded successfully!")
    print(f"   Device: {DEVICE}")
    print(f"   Model size: ~{sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")

Loading Qwen/Qwen2.5-0.5B-Instruct...
✅ Model loaded successfully!
   Device: mps
   Model size: ~0.5B parameters


<a id='generation'></a>

# Text Generation

In this section, we'll define a helper function to generate text locally using Qwen.

This process is called **inference** — it’s when we give the model an input prompt and let it predict the next words, one token at a time.

The function we’ll create will handle three main tasks:

1. **Formatting the input** — turning a text prompt into tokens the model understands.  
2. **Generating new tokens** — asking the model to produce text based on the prompt.  
3. **Decoding the output** — converting tokens back into readable text.

In [23]:
def generate_text(prompt, max_new_tokens=100, temperature=0.7):
    """Generate a clean response from Qwen."""
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    # Convert messages into a model-friendly text format using the tokenizer's chat template.
    # `add_generation_prompt=True` tells the tokenizer to add the "assistant" prefix automatically.
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Convert the formatted text into input tensors (numerical form the model can process)
    # and move them to the correct device (CPU or GPU).
    inputs = tokenizer(formatted, return_tensors="pt").to(DEVICE)

    # Turn off gradient tracking since we're not training the model (only generating text).
    with torch.no_grad():
        output = model.generate(
            **inputs,                             # Feed input tensors into the model
            max_new_tokens=max_new_tokens,        # Limit output length
            temperature=temperature,              # Control creativity / variability
            do_sample=temperature > 0,            # Enable sampling only if temperature > 0
            top_p=0.7,                            # Nucleus sampling: model only considers top 70% likely tokens
            repetition_penalty=1.1,               # Slightly penalize repeating words
            pad_token_id=tokenizer.pad_token_id,  # Avoid padding issues
        )

    # Decode only the newly generated tokens (skip the input prompt portion)
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]

    # Convert the generated tokens back to readable text, skipping special tokens like <EOS>.
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Return the clean, final string
    return response

## Test Generation

Let's generate a text using our local model.

In [24]:
prompt = "What are the main benefits of renewable energy?"

print(f"Model: {MODEL_NAME}")
print(f"Prompt: {prompt}\n")
print("Generating response...")

import time
start = time.time()
response = generate_text(prompt, max_new_tokens=150, temperature=0.7)
elapsed = time.time() - start

print("\nResponse:")
print("-" * 50)
print(response)
print("-" * 50)
print(f"\n⏱️ Generation time: {elapsed:.2f}s  (~{150/elapsed:.1f} tok/s)")

Model: Qwen/Qwen2.5-0.5B-Instruct
Prompt: What are the main benefits of renewable energy?

Generating response...

Response:
--------------------------------------------------
Renewable energy sources, such as solar, wind, and hydroelectric power, offer several significant advantages that contribute to reducing our dependence on fossil fuels and mitigating climate change.

1. **Sustainability**: Renewable energy sources like solar and wind do not deplete natural resources or have negative environmental impacts. They can be replenished naturally over time with new installations.

2. **Environmental Impact**: These energy sources produce little to no greenhouse gas emissions during operation, making them ideal for combating global warming. Additionally, they often generate clean air and water, which is beneficial for human health and ecosystems.

3. **Energy Independence**: By relying less on imported fossil fuels, countries can reduce their dependency on foreign oil and gas supplies, th

## Multiple Prompts

We can prompt using a simple loop to get a bunch of responses:

In [6]:
# Process multiple prompts efficiently
prompts = [
    "Explain machine learning in simple terms",
    "What are the causes of climate change?",
    "How does social media affect society?"
]

for i, prompt in enumerate(prompts, 1):
    print(f"Prompt {i}: {prompt}")
    response = generate_text(prompt, max_new_tokens=100, temperature=0.7)
    print(f"Response: {response}\n")
    print("-" * 50)

Prompt 1: Explain machine learning in simple terms
Response: Machine learning is a type of artificial intelligence that allows computers to learn and improve from data without being explicitly programmed. It involves training algorithms on large datasets, which allow the computer to identify patterns and make predictions or decisions based on those patterns.

In simple terms, it's like teaching a machine how to do things by showing it examples of what it should do. The more examples it sees, the better it can learn and become more accurate at predicting outcomes. This process is repeated over and over again until the

--------------------------------------------------
Prompt 2: What are the causes of climate change?
Response: Climate change is primarily caused by human activities, including burning fossil fuels and deforestation. These activities release large amounts of greenhouse gases into the atmosphere, which trap heat in the Earth's atmosphere and cause the planet to warm up. Add

## Different Temperature Settings

Theoretical Max: No hard limit! You can set temperature to 10, 100, or even 1000.

Practical Max: Usually 1.5-2.0 is the useful limit.

In [7]:
# Compare different temperature settings
prompt = "Complete this sentence: 'Happiness is like a"
temperatures = [0.3, 1.0, 2.0, 5.0]

print(f"Testing temperature effects\n")
print(f"Prompt: {prompt}\n")

for temp in temperatures:
    print(f"Temperature {temp}:")
    response = generate_text(prompt, max_new_tokens=80, temperature=temp)
    print(f"{response}\n")

Testing temperature effects

Prompt: Complete this sentence: 'Happiness is like a

Temperature 0.3:
joyful breeze that brings joy and light to our lives, making us feel uplifted and fulfilled.'

Temperature 1.0:
honey, that's fleeting and can only be enjoyed once.'

Temperature 2.0:
bottle of champagne, overflowing with life and joy, making one feel rejuvenated and fulfilled.

Temperature 5.0:
joyful smile painted on her face when feeling overwhelmed. ' Happiness is likenified with this gentle and radiant light in people, particularly at those experiencing great difficulties that make every other feeling look waxy pale in comparison. A truly joy-filled expression, the cheerful, optimistic aspect evinces something profound about our humanity that deftly captures a vast world's potential to embrace peace and fulfillment in everyday experiences



Smaller models have less diverse "creativity" - they've learned fewer patterns, so they default to common metaphors.

### Temp = 0?

When temperature = 0, the model stops sampling from a probability distribution and instead always picks the single most likely next token — this is called greedy decoding.

That means the output is completely deterministic: the same prompt will always produce the exact same response.
It’s useful when you want precise, repeatable answers (e.g., for testing or structured output), but it removes creativity and variation that come from randomness at higher temperatures.

In [29]:
# Define a simple prompt
prompt = "Explain the difference between supervised and unsupervised learning."

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

# Function to generate deterministic output
def greedy_generate(inputs):
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,       # Greedy decoding (no randomness)
            pad_token_id=tokenizer.pad_token_id
        )
    # Decode only the new text
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

# Run twice
output_1 = greedy_generate(inputs)
output_2 = greedy_generate(inputs)

print("Run 1:\n")
print(output_1)
print("\n" + "-" * 80 + "\n")
print("Run 2:\n")
print(output_2)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Run 1:

Explain the steps involved in each type of learning.

Supervised Learning:
In supervised learning, we have a labeled dataset where the input data is paired with an output label that indicates the correct answer to the problem. The goal is to learn a model that can predict the output based on the input data. There are two main types of supervised learning: classification and regression.
Classification involves predicting one of several possible classes for an input data point. For example, if we want to classify emails as spam or not spam, we would use a classifier to determine whether an email is spam or not spam.
Regression involves

--------------------------------------------------------------------------------

Run 2:

Explain the steps involved in each type of learning.

Supervised Learning:
In supervised learning, we have a labeled dataset where the input data is paired with an output label that indicates the correct answer to the problem. The goal is to learn a model tha

## Explore Probabilities

In [8]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def get_token_probabilities(prompt, temperature=1.0):
    """Get probability distribution for next token"""
    
    # Tokenize - returns PyTorch tensors
    inputs = tokenizer(prompt, return_tensors="pt")

    # move to GPU/Apple Silicon if available
    if DEVICE != "cpu":
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    # Get model output (raw logits)
    with torch.no_grad():                   # Don't calculate gradients (save memory)
        outputs = model(**inputs)           # Run the model forward pass
        logits = outputs.logits[0, -1, :]   # Last token's predictions. 0 = batch item, -1 = last position in seq (after "a"), : = all vocab tokens
        
    # Apply temperature - this is what a model does internally
    logits_with_temp = logits / temperature
    
    # Convert to probabilities with softmax 
    probs = F.softmax(logits_with_temp, dim=-1)
    
    # Get top tokens
    top_k = 20
    top_probs, top_indices = torch.topk(probs, top_k)
    
    # Decode tokens back to text
    tokens = [tokenizer.decode([idx.item()]) for idx in top_indices]
    
    return tokens, top_probs.cpu().numpy(), logits.cpu().numpy()

# Analyze a prompt
prompt = "Complete this sentence: 'Happiness is like a"
tokens, probs, raw_logits = get_token_probabilities(prompt, temperature=1.0)

# Display results
print(f"Top 20 token probabilities for: '{prompt}'\\n")
for token, prob in zip(tokens[:10], probs[:10]):
    bar = "█" * int(prob * 100)
    print(f"{token:15s} {prob:.4f} {bar}")

Top 20 token probabilities for: 'Complete this sentence: 'Happiness is like a'\n
 flower         0.0864 ████████
 beautiful      0.0728 ███████
 warm           0.0266 ██
 river          0.0202 ██
 butterfly      0.0185 █
 sunny          0.0122 █
 rainbow        0.0111 █
 ______         0.0109 █
 snow           0.0103 █
 well           0.0090 


## Using Templated prompts

In [10]:
topics = ["remote work", "artificial intelligence", "meditation"]

for topic in topics:
    prompt = f"Write a brief summary about {topic}:"
    print(f"\nSummary for {topic}:")
    output = generate_text(prompt, max_new_tokens=100, temperature=0.5)
    print(output)



Summary for remote work:
Remote work involves working from home or using technology to access the internet and collaborate with colleagues, regardless of their physical location. This can be beneficial for many reasons, including flexibility, cost savings, and improved productivity. However, it also requires effective communication skills, time management, and adaptability to new work environments.

Summary for artificial intelligence:
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI is characterized by its ability to learn from data, adapt to new situations, and make decisions based on information. It has been used in a wide range of applications, including voice assistants like Siri and Alexa, self-driving cars, and natural language processing for language translation and text generation. The field of

# Using LLMs for Research

Large Language Models (LLMs) can support computational social science by helping researchers interpret, classify, or summarize complex text data at scale.

In [14]:
import pandas as pd
import random

# Load the dataset
df = pd.read_csv("../../data/aita_top_subs.csv")

# Sample a few posts to analyze
sample_texts = random.sample(df["selftext"].dropna().tolist(), 3)

# Define a reasoning-style prompt
prompt_template = """
        You are a tool for evaluating moral dilemmas.

        Please evaluate the following post from the subreddit "Am I the Asshole".
        
        <dilemma>
        {text}
        </dilemma>
        
        First, determine whether the OP (original poster) of this post is the asshole in the scenario they describe.
        Provide a categorical label indicating your judgment of the scenario, from one of these choices:

        - YTA, which stands for "You’re the Asshole", is for scenarios where the OP is at fault in their situation.
        - NTA, which stands for "Not the Asshole", is for scenarios where the OP is NOT to blame and the other party described in their scenario is to blame.
        - ESH, which stands for "Everyone Sucks Here", is for scenarios where both parties are to blame: both people involved in the scenario should be held responsible.
        - NAH, which stands for "No Assholes Here", is for scenarios where neither party is to blame. All parties actions are justified. Nobody needs to be held accountable. Shit happens.
        - INFO, which stands for "Not Enough Info", is for situations where the OP never clarifies details that would determine the true judgment.

        Then, please provide an explanation for why you chose this label. Restrict your explanation to ONE paragraph.

"""

# Run inference on a few samples
for i, text in enumerate(sample_texts, 1):
    print(f"\nExample {i}")
    prompt = prompt_template.format(text=text[:1000])  # truncate to avoid token limits
    response = generate_text(prompt, max_new_tokens=200, temperature=0.7)
    print("-" * 50)
    print(response)


Example 1
--------------------------------------------------
The original poster is clearly expressing dissatisfaction with their mother's decision to allow them to purchase a TV that they consider unnecessary or impractical for their living space. This behavior suggests a lack of self-awareness regarding their financial constraints and a desire to prioritize their family over personal desires. The scenario presents a clear case of a parent's unreasonable demand leading to conflict and resentment, making it a classic example of a "Not the Asshole" scenario.

In this case:
1. **Categorical Label**: NTA
2. **Explanation**: The scenario highlights a significant issue with parental authority and child autonomy. The original poster expresses frustration with their mother's decision, implying that they feel like they're being taken advantage of by their parents' wishes rather than their own. This reflects a deep-seated sense of inadequacy and entitlement, which is characteristic of an "Not 

## Using Structured Output (JSON)

By prompting models to return structured JSON outputs that follow a fixed schema (validated with tools like Pydantic), we can transform qualitative social media data—like moral reasoning in r/AmItheAsshole posts—into analyzable, reproducible datasets.

In [ ]:
import pandas as pd
import json
import random

# Reusable JSON instruction string
JSON_INSTRUCTIONS = {
    "aita": """
    Your response must be a single JSON object with exactly two keys: "judgment" and "explanation".
    {
    "judgment": "YTA|NTA|ESH|NAH|INFO",
    "explanation": "A clear explanation of why you chose this judgment"
    }
    Do not include any additional text, markdown formatting, or commentary.
    """
}

def analyze_aita_post(text):
    prompt = f"""
    You are analyzing moral judgments in Reddit posts from r/AmItheAsshole (AITA).
    Read the post below and decide who is at fault.
    Follow these exact instructions:
{JSON_INSTRUCTIONS['aita']}

Post:
"{text}"
"""
    response = generate_text(prompt, temperature=0.5, max_new_tokens=300)

    # Clean up code fences
    cleaned = (
        response.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    try:
        data = json.loads(cleaned)
        return data
    except:
        # Fallback if model output is messy
        return {"judgment": "INFO", "explanation": cleaned[:200]}

In [22]:
# Run the analysis
post = sample_texts[0]
result = analyze_aita_post(post)

# Display
print("Full AITA Post:\n")
print(post.strip())
print("\n--------------------------------------------------")
print("Model Output:")
print(json.dumps(result, indent=2))

Full AITA Post:

I'm (20M) in the market for a new TV, the 32 inch Sharp TV I have still runs fine but I want something bigger, I have my eyes on a 65 inch LG Smart TV, I saved up for it and had plans to buy it but my mom told me that if I bought a TV when I already have a perfectly working TV then I'd have to put it in the living room so that everyone (my parents and sibs) can enjoy it.

She said that it was selfish and that I don't need a TV that big for my room especially when I already have one. I said fine and a couple weeks passed and she asked me when I'm buying the TV, I told her that I'm not buying one anymore and she asked why and I said there's no point for me to spend my hard-earned money for a TV that I'm not even allowed to put in my own room.

She went on a tirade about how I'm the most selfish person in the world and that I was selfish to buy a TV in the first place with the intention of putting it in my room and that now the only reason I'm not buying the TV is because

## Tips for Local Inference

1. **Model Selection**
   - Start small: Test with 1-3B models first
   - Match to RAM: ~2GB RAM per billion parameters
   - Quality vs Speed: Larger models are better but slower

2. **Performance Optimization**
   - Close other applications to free RAM
   - Reduce max_new_tokens for faster responses
   - Lower temperature for more focused (faster) generation

3. **When to Use Local vs Cloud**
   - **Local**: Privacy-sensitive data, offline work, no usage limits
   - **Cloud**: Need larger models, faster inference, GPU acceleration

---

## Stretch Goals 

With Hugging Face’s transformers library, you can try out a variety of pretrained and fine-tuned models. If you finish early, explore some of these challenges:

1. Sentiment Analysis → Analyze the sentiment of AITA posts. (Hint: distilbert-base-uncased-finetuned-sst-2-english)
2. Text Classification → Classify Reddit posts by topic or category. (Hint: search Hugging Face for “text classification”)
3. Question Answering → Ask questions about an AITA post and see if the model can extract an answer. (Hint: deepset/roberta-base-squad2)
4. Summarization → Generate concise summaries of posts. (Hint: facebook/bart-large-cnn)
5. Translation → Try translating posts into another language. (Hint: Helsinki-NLP opus-mt models)

In [ ]:
# Space for stretch goals
